In [ ]:
import json
#Importing the pandas Library 
import pandas as pd 

### Read Parquet File

In [ ]:
# Open and read the JSON file
with open('recipes_output.json', 'rb') as file:
    data = json.load(file)

In [154]:
data

[{'name': 'Lentil, Apple, and Turkey Wrap',
  'instructions': '1. Place the stock, lentils, celery, carrot, thyme, and salt in a  saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool. 2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper. 3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.',
  'ingredients': [{'name': 'low-sodium vegetable or chicken stock',
    'quantity': 4.0,
    

In [155]:
# Convert to a DataFrame
df = pd.json_normalize(data)

# Create a new column 'ingredient_names' that contains a list of ingredient names
df['ingredient_names'] = df['ingredients'].apply(lambda ingredients: [ingredient['name'] for ingredient in ingredients])

# Drop the original 'ingredients' column if you only need the list of names
df = df.drop(columns=['ingredients'])
df = df.drop(columns=['calories'])
df = df.drop(columns=['fat'])
df = df.drop(columns=['protein'])
df = df.drop(columns=['desc'])

In [157]:
df.columns

Index(['name', 'instructions', 'total_time', 'ingredient_names'], dtype='object')

In [158]:
df['name'] = df['name'].str.replace('"', '', regex=False)

In [159]:
def count_empty_ingredient_lists(df):
    # Count the number of empty lists in the 'ingredient_names' column
    empty_list_count = df['ingredient_names'].apply(lambda x: x == []).sum()
    return empty_list_count

In [160]:
count_empty_ingredient_lists(df)

76

In [161]:
def remove_empty_ingredient_rows(df):
    # Filter out rows where 'ingredient_names' is an empty list
    filtered_df = df[df['ingredient_names'].apply(lambda x: len(x) > 0)]
    return filtered_df

In [162]:
filtered_df = remove_empty_ingredient_rows(df)

In [163]:
filtered_df

,name,instructions,total_time,ingredient_names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...",30.0,"[low-sodium vegetable or chicken stock, dried ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,177.0,"[whipping cream, onions, chopped, salt, bay le..."
2,Potato and Fennel Soup Hodge,In a heavy saucepan cook diced fennel and oni...,40.0,"[fennel bulb (sometimes called anise), stalks ..."
3,Mahi-Mahi in Tomato Olive Sauce,Heat oil in heavy skillet over -high heat. Ad...,22.0,"[extra-virgin olive oil, chopped onion, dry wh..."
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,55.0,"[12-ounce package frozen spinach soufflé, tha..."
...,...,...,...,...
20125,Parmesan Puffs,Beat whites in a bowl with an electric mixer u...,3.0,"[egg whites, Parmigiano-Reggiano, finely grate..."
20126,Artichoke and Parmesan Risotto,Bring broth to simmer in saucepan.Remove from ...,41.0,"[(or more) low-salt chicken broth, butter, div..."
20127,Turkey Cream Puff Pie,"Using a sharp knife, cut a shallow X in bottom...",67.0,"[tomato, onion, finely chopped (1/2 cup), unsa..."
20128,Snapper on Angel Hair with Citrus Cream,Heat 2 tablespoons oil in heavy skillet over ...,14.0,"[olive oil, shallots, thinly sliced (about 1/2..."


In [167]:
def remove_duplicates_by_column(df, column):
    # Identify exact duplicate rows based on the 'name' column
    exact_duplicates = df[df.duplicated(subset=column, keep=False)]
    
    # Print the exact duplicates
    print(f"Exact duplicate rows based on {column}:")
    print(exact_duplicates)
    
    # Remove exact duplicates based on the 'name' column, keeping only the first occurrence
    filtered_df = df.drop_duplicates(subset=column, keep='first')
    
    return filtered_df

In [ ]:
filtered_df_name = remove_duplicates_by_column(df, 'name')

Exact duplicate rows based on name:
                                                    name  \
14                                         Peach Mustard   
17                           Crisp Braised Pork Shoulder   
21                                         Fried Chicken   
24                               Sea Salt-Roasted Pecans   
25                                Garlic Baguette Crumbs   
27                                     Dried Pear Crisps   
31                       Moroccan-Style Preserved Lemons   
36           Pastry Twists with Spiced Sugar-Honey Glaze   
37                                 Sauteed Broccoli Rabe   
39                          Better-Than-Pita Grill Bread   
43                         Purple-Potato and Crab Gratin   
45                                    Pickled Red Onions   
57                                       Pumpkin Muffins   
58                                 Orange Balsamic Glaze   
62     Southwest Corn Bread Stuffing with Corn and Gr...   
67  

In [169]:
filtered_df_name

,name,instructions,total_time,ingredient_names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...",30.0,"[low-sodium vegetable or chicken stock, dried ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,177.0,"[whipping cream, onions, chopped, salt, bay le..."
2,Potato and Fennel Soup Hodge,In a heavy saucepan cook diced fennel and oni...,40.0,"[fennel bulb (sometimes called anise), stalks ..."
3,Mahi-Mahi in Tomato Olive Sauce,Heat oil in heavy skillet over -high heat. Ad...,22.0,"[extra-virgin olive oil, chopped onion, dry wh..."
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,55.0,"[12-ounce package frozen spinach soufflé, tha..."
...,...,...,...,...
20124,Chinese Barbecued Spareribs,Wash spareribs. Remove excess fat and gristle....,30.0,"[side pork spareribs, about 3 pounds, soy sauc..."
20126,Artichoke and Parmesan Risotto,Bring broth to simmer in saucepan.Remove from ...,41.0,"[(or more) low-salt chicken broth, butter, div..."
20127,Turkey Cream Puff Pie,"Using a sharp knife, cut a shallow X in bottom...",67.0,"[tomato, onion, finely chopped (1/2 cup), unsa..."
20128,Snapper on Angel Hair with Citrus Cream,Heat 2 tablespoons oil in heavy skillet over ...,14.0,"[olive oil, shallots, thinly sliced (about 1/2..."


In [172]:
filtered_df_all = remove_duplicates_by_column(filtered_df_name, 'instructions')

Exact duplicate rows based on instructions:
                                                    name  \
137                                  Honey Mustard Sauce   
348                           Strawberry Banana Smoothie   
453                                   Ginger Spice Syrup   
527                                                 Tago   
528                                          Bee's Knees   
547                                        Blue Mountain   
577                                      Orange Gin Fizz   
590                                             Affinity   
758                                          Margarita I   
770                                         Moulin Rouge   
951                                 Mixed-Berry Daiquiri   
1076                                             Unknown   
1453                           Barley and Mushroom Pilaf   
1466                                   Maiden's Prayer I   
1484                             Mozzarella Pesto Spread

In [173]:
filtered_df_all

,name,instructions,total_time,ingredient_names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...",30.0,"[low-sodium vegetable or chicken stock, dried ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,177.0,"[whipping cream, onions, chopped, salt, bay le..."
2,Potato and Fennel Soup Hodge,In a heavy saucepan cook diced fennel and oni...,40.0,"[fennel bulb (sometimes called anise), stalks ..."
3,Mahi-Mahi in Tomato Olive Sauce,Heat oil in heavy skillet over -high heat. Ad...,22.0,"[extra-virgin olive oil, chopped onion, dry wh..."
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,55.0,"[12-ounce package frozen spinach soufflé, tha..."
...,...,...,...,...
20124,Chinese Barbecued Spareribs,Wash spareribs. Remove excess fat and gristle....,30.0,"[side pork spareribs, about 3 pounds, soy sauc..."
20126,Artichoke and Parmesan Risotto,Bring broth to simmer in saucepan.Remove from ...,41.0,"[(or more) low-salt chicken broth, butter, div..."
20127,Turkey Cream Puff Pie,"Using a sharp knife, cut a shallow X in bottom...",67.0,"[tomato, onion, finely chopped (1/2 cup), unsa..."
20128,Snapper on Angel Hair with Citrus Cream,Heat 2 tablespoons oil in heavy skillet over ...,14.0,"[olive oil, shallots, thinly sliced (about 1/2..."


In [ ]:
filtered_df_all.to_parquet('deduplicated_recipes_Epicurious.parquet', index=False)

print("Deduplication complete! Results saved to:")
print(" - deduplicated_recipes_Epicurious.parquet (deduplicated data)")